# Experiment 1: Variation in Productivity

In [1]:
import os
from pathlib import Path

# Import AUTO-07p command interface
from AUTOclui import AUTOCommands as ac 
from AUTOclui import runAUTO as ra

import matplotlib.pyplot as plt

from plot_3x3 import plot_equilibrium_diagram, save_3x3_plot

from pyvirtualdisplay import Display

In [2]:
folder = Path.cwd()
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output'
output_folder.mkdir(exist_ok=True)

parameter_file = folder / 'experiment_parameters.dat'

# Values used in the simple nine-panel loop
movement_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 1, 3, 6, 10]
driver_index = 15
driver_limits = [0.0, 1.0]
start_deltar = 0.0
start_deltas = 0.0

In [3]:
# Start a hidden display so AUTO can run without a desktop session
display = Display(visible=False, size=(1200, 900))
display.start()

In [4]:
# Repeat the original single-plot continuation for each movement rate
runner = ra.runAUTO()
movement_panels = []

for movement_rate in movement_rates:
    print(f'\nMovement rate is {movement_rate:g}')

    # Tell common_model.f90 which habitat contrasts and movement rate to use
    parameter_file.write_text(
        f'{start_deltar} {start_deltas} {movement_rate}\n'
    )

    # First reproduce the ordinary one-parameter equilibrium continuation
    eq_forward = ac.run(
        e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200
    )
    eq_backward = ac.run(
        eq_forward('EP1'), DS=-1.0e-2, runner=runner,
        NMX=4000, NPR=200, UZSTOP={14: -0.25},
    )
    equilibrium = (eq_forward + eq_backward).relabel()
    # Fully load the two restart points before two-parameter continuation
    bp_start = equilibrium('BP1')
    lp_start = equilibrium('LP1')
    bp_start.PAR.toarray()
    lp_start.PAR.toarray()

    # Continue only BP1: the recovery-threshold curve
    bp_curve = ac.run(
        bp_start,
        ICP=[14, driver_index], ISW=2,
        DS=-1.0e-3, DSMIN=1.0e-5, DSMAX=5.0e-3,
        NMX=8000, NPR=400,
        UZSTOP={14: [-0.25, 0.5], driver_index: driver_limits},
        runner=runner,
    )

    # Continue only LP1: the collapse-threshold curve
    lp_curve = ac.run(
        lp_start,
        ICP=[14, driver_index], ISW=2,
        DS=1.0e-3, DSMIN=1.0e-5, DSMAX=5.0e-3,
        NMX=8000, NPR=400,
        UZSTOP={14: [-0.25, 0.5], driver_index: driver_limits},
        runner=runner,
    )

    movement_panels.append({
        'movement_rate': movement_rate,
        'bp_curve': bp_curve,
        'lp_curve': lp_curve,
    })

# One additional equilibrium run supplies the Pelagic Predator 1D figure
parameter_file.write_text(f'{start_deltar} {start_deltas} 2.0\n')
reference_forward = ac.run(
    e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200
)
reference_backward = ac.run(
    reference_forward('EP1'), DS=-1.0e-2, runner=runner,
    NMX=4000, NPR=200, UZSTOP={14: -0.25},
)
reference_equilibrium = (reference_forward + reference_backward).relabel()


Movement rate is 0.1
gfortran -g -fopenmp -O -c common_model.f90 -o common_model.o
gfortran -g -fopenmp -O common_model.o -o common_model.exe /auto/lib/*.o
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1    10  BP    2   1.14286E-01   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00
   1    18  UZ    3   5.00000E-01   9.42809E+00   0.00000E+00   6.66667E+00   0.00000E+00   0.00000E+00   6.66667E+00   0.00000E+00

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   2    47  LP    4   1.32335E-01   7.71689E+00   3.27910E-01   5.43997E+00   2.72857E-01   3.27910E-01   5.43997E+00   2.72857E-01
   2   200        5  

In [5]:
# Plot pelagic predator abundance against fishing effort for the reference run
fig = plot_equilibrium_diagram(reference_equilibrium)
fig.savefig(
    output_folder / '1d_experiment_1_productivity.png',
    dpi=200,
    bbox_inches='tight',
)
# Uncomment to also save an editable vector version
# fig.savefig(output_folder / '1d_experiment_1_productivity.svg', bbox_inches='tight')
plt.close(fig)

In [6]:
# Create and save the 3x3 figure
save_3x3_plot(
    movement_panels,
    experiment=1,
    output_folder=output_folder,
    # save_svg=True,  # Uncomment to also save an SVG file
)

In [7]:
# Remove AUTO working files and stop the hidden display
runner.config(clean=True)
ac.clean()
display.stop()
parameter_file.unlink(missing_ok=True)

Deleting fort.* *.o *.exe *.*~ ... done
